In [1]:
# ============================================================
# ERIP - Macroeconomic Scenario Bronze Ingestion
# Notebook: nb_ingest_macroeconomic
# Purpose:
# 1. Read Macroeconomic Scenario source file
# 2. Validate data contract
# 3. Run data quality checks
# 4. Write Bronze Delta table
# 5. Log ingestion metadata and DQ results
# ============================================================

# ====================================================
# SECTION 1
# Pipeline Initialization
# ====================================================

from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime

source_system = "Macroeconomic Scenario Feed"
source_file_path = "Files/01_Bronze/macroeconomic_data/macroeconomic_data.csv"
target_table = "bronze_macroeconomic_data"
pipeline_name = "nb_ingest_macroeconomic"

run_start_time = datetime.now()

print("ERIP Macroeconomic Scenario ingestion started")
print(f"Source system: {source_system}")
print(f"Source file: {source_file_path}")
print(f"Target table: {target_table}")

StatementMeta(, a8e3968a-3b15-4273-ae12-7c10b8fbc572, 3, Finished, Available, Finished, False)

ERIP Macroeconomic Scenario ingestion started
Source system: Macroeconomic Scenario Feed
Source file: Files/01_Bronze/macroeconomic_data/macroeconomic_data.csv
Target table: bronze_macroeconomic_data


In [2]:
# ====================================================
# SECTION 2
# Read source CSV
# ====================================================

macro_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(source_file_path)
)

display(macro_df.limit(10))

print(f"Rows read: {macro_df.count()}")
print(f"Columns read: {len(macro_df.columns)}")

StatementMeta(, a8e3968a-3b15-4273-ae12-7c10b8fbc572, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8d8a332c-e70c-41c1-8947-3705f9c41a30)

Rows read: 108
Columns read: 14


In [3]:
# ============================================================
# SECTION 3 - DATA CONTRACT VALIDATION
# ============================================================

required_columns = [
    "scenario_id",
    "scenario_name",
    "scenario_severity",
    "month",
    "gdp_growth_pct",
    "inflation_pct",
    "interest_rate_pct",
    "unemployment_pct",
    "commercial_property_price_index",
    "credit_spread_bps",
    "pd_stress_multiplier",
    "lgd_stress_multiplier",
    "source_system",
    "extract_date"
]

missing_columns = list(
    set(required_columns) -
    set(macro_df.columns)
)

if len(missing_columns) == 0:
    print("✓ Data Contract Validation Passed")
else:
    print("✗ Missing Columns:")
    print(missing_columns)

StatementMeta(, a8e3968a-3b15-4273-ae12-7c10b8fbc572, 5, Finished, Available, Finished, False)

✓ Data Contract Validation Passed


In [4]:
# ============================================================
# SECTION 4 - THREE-LAYER DATA CONTRACT VALIDATION
# Layer 1: Schema Rules
# Layer 2: Business Rules
# Layer 3: Regulatory / Banking Rules
# ============================================================

validation_results = []

def add_validation_result(layer, rule_id, rule_name, failed_count):
    status = "PASS" if failed_count == 0 else "FAIL"
    validation_results.append({
        "pipeline_name": pipeline_name,
        "source_system": source_system,
        "target_table": target_table,
        "validation_layer": layer,
        "rule_id": rule_id,
        "rule_name": rule_name,
        "failed_count": int(failed_count),
        "status": status,
        "validation_timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    })

# ----------------------------
# Layer 1: Schema Rules
# ----------------------------
add_validation_result(
    "Schema",
    "SCHEMA_001",
    "All required columns must be present",
    len(missing_columns)
)

# ----------------------------
# Layer 2: Business Rules
# ----------------------------
total_rows = macro_df.count()

duplicate_scenario_ids = total_rows - macro_df.select("scenario_id").distinct().count()
null_scenario_ids = macro_df.filter(col("scenario_id").isNull()).count()
null_months = macro_df.filter(col("month").isNull()).count()
invalid_scenario_name = macro_df.filter(
    ~col("scenario_name").isin("Baseline", "Adverse", "Severe")
).count()
invalid_scenario_severity = macro_df.filter(
    ~col("scenario_severity").isin("Low", "Medium", "High")
).count()

add_validation_result("Business", "BUS_001", "Scenario ID must be unique", duplicate_scenario_ids)
add_validation_result("Business", "BUS_002", "Scenario ID must not be null", null_scenario_ids)
add_validation_result("Business", "BUS_003", "Scenario month must not be null", null_months)
add_validation_result("Business", "BUS_004", "Scenario name must be valid", invalid_scenario_name)
add_validation_result("Business", "BUS_005", "Scenario severity must be valid", invalid_scenario_severity)

# ----------------------------
# Layer 3: Regulatory / Banking Rules
# ----------------------------
invalid_gdp = macro_df.filter((col("gdp_growth_pct") < -20) | (col("gdp_growth_pct") > 20)).count()
invalid_inflation = macro_df.filter((col("inflation_pct") < -5) | (col("inflation_pct") > 25)).count()
invalid_interest_rate = macro_df.filter((col("interest_rate_pct") < 0) | (col("interest_rate_pct") > 20)).count()
invalid_unemployment = macro_df.filter((col("unemployment_pct") < 0) | (col("unemployment_pct") > 30)).count()
invalid_property_index = macro_df.filter(col("commercial_property_price_index") <= 0).count()
invalid_credit_spread = macro_df.filter(col("credit_spread_bps") < 0).count()
invalid_pd_multiplier = macro_df.filter(col("pd_stress_multiplier") <= 0).count()
invalid_lgd_multiplier = macro_df.filter(col("lgd_stress_multiplier") <= 0).count()

add_validation_result("Scenario", "REG_001", "GDP growth must be within stress-testing range", invalid_gdp)
add_validation_result("Scenario", "REG_002", "Inflation must be within stress-testing range", invalid_inflation)
add_validation_result("Scenario", "REG_003", "Interest rate must be within stress-testing range", invalid_interest_rate)
add_validation_result("Scenario", "REG_004", "Unemployment must be within stress-testing range", invalid_unemployment)
add_validation_result("Scenario", "REG_005", "Commercial property index must be positive", invalid_property_index)
add_validation_result("Scenario", "REG_006", "Credit spread must not be negative", invalid_credit_spread)
add_validation_result("Scenario", "REG_007", "PD stress multiplier must be positive", invalid_pd_multiplier)
add_validation_result("Scenario", "REG_008", "LGD stress multiplier must be positive", invalid_lgd_multiplier)

validation_df = spark.createDataFrame(validation_results)

display(validation_df)

failed_validations = validation_df.filter(col("status") == "FAIL").count()

if failed_validations > 0:
    raise Exception(f"Macroeconomic Scenario Validation Failed: {failed_validations} validation rule(s) failed.")
else:
    print("✓ Macroeconomic Scenario Three-layer Validation Passed")

StatementMeta(, a8e3968a-3b15-4273-ae12-7c10b8fbc572, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 283879bd-4ddd-47aa-b03e-4d5566b0d5b8)

✓ Macroeconomic Scenario Three-layer Validation Passed


In [5]:
# ============================================================
# SECTION 4 - VALIDATION FRAMEWORK
# Data Contract + Data Quality + Business + Regulatory Checks
# ============================================================

# Add ingestion audit columns to source data
macro_bronze_df = (
    macro_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("pipeline_name", lit(pipeline_name))
    .withColumn("bronze_load_date", current_date())
)

# Write Bronze Delta table
macro_bronze_df.write.mode("overwrite").format("delta").saveAsTable("bronze_macroeconomic_data")

# Write validation results table
validation_df.write.mode("append").format("delta").saveAsTable("dq_validation_results")

print("✓ Bronze Delta table created: bronze_customer_master")
print("✓ DQ validation results written: dq_validation_results")
print(f"Rows written to Bronze: {macro_bronze_df.count()}")

StatementMeta(, a8e3968a-3b15-4273-ae12-7c10b8fbc572, 7, Finished, Available, Finished, False)

✓ Bronze Delta table created: bronze_customer_master
✓ DQ validation results written: dq_validation_results
Rows written to Bronze: 108


In [6]:
# ============================================================
# SECTION 5 - DATA QUALITY SUMMARY
# ============================================================

total_validation_rules = validation_df.count()
passed_validation_rules = validation_df.filter(col("status") == "PASS").count()
failed_validation_rules = validation_df.filter(col("status") == "FAIL").count()

dq_score = (passed_validation_rules / total_validation_rules) * 100

print("Data Quality Summary")
print("--------------------")
print(f"Total validation rules: {total_validation_rules}")
print(f"Passed validation rules: {passed_validation_rules}")
print(f"Failed validation rules: {failed_validation_rules}")
print(f"Data Quality Score: {dq_score}%")

StatementMeta(, a8e3968a-3b15-4273-ae12-7c10b8fbc572, 8, Finished, Available, Finished, False)

Data Quality Summary
--------------------
Total validation rules: 14
Passed validation rules: 14
Failed validation rules: 0
Data Quality Score: 100.0%


In [7]:
# ============================================================
# SECTION 6 - METADATA LOGGING
# ============================================================

from datetime import datetime

run_end_time = datetime.now()
execution_time_seconds = (run_end_time - run_start_time).total_seconds()

metadata = [{
    "pipeline_name": pipeline_name,
    "source_system": source_system,
    "target_table": target_table,
    "rows_processed": macro_bronze_df.count(),
    "validation_rules": total_validation_rules,
    "dq_score": dq_score,
    "status": "SUCCESS",
    "run_start_time": run_start_time.strftime("%Y-%m-%d %H:%M:%S"),
    "run_end_time": run_end_time.strftime("%Y-%m-%d %H:%M:%S"),
    "execution_time_seconds": execution_time_seconds
}]

metadata_df = spark.createDataFrame(metadata)

metadata_df.write \
    .mode("append") \
    .format("delta") \
    .saveAsTable("metadata_ingestion_log")

display(metadata_df)

print("✓ Metadata successfully written")

StatementMeta(, a8e3968a-3b15-4273-ae12-7c10b8fbc572, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2d092f32-45e9-41ca-a572-c8dec153adf7)

✓ Metadata successfully written
